# GNNHAR on S&P 500 (Colab A100)

Faithful GNNHAR (Zhang, Pu, Cucuringu & Dong, IJF 2025, arXiv:2308.01419) on the S&P 500 full-matrix protocol: daily Parkinson variance, horizons {1,5,10,22}, QLIKE + date-clustered Diebold-Mariano, vs HAR / own-history gamma-GBM / GBM+corr-graph / no-graph. GNNHAR is GPU-bound (pure PyTorch), so A100 speeds it up over a local RTX 4060.

Workflow: git-clone the repo (driver + deps committed on master), pull SP500 DATA from Drive (colab_bundle_sp500_clean.zip), run, then DOWNLOAD the result JSON. Set runtime to A100 GPU.

In [ ]:
# 0. GPU (want A100)
get_ipython().system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

In [ ]:
# 1. Clone CODE from master (has scripts/eda/gnnhar_sp500.py + full_matrix.py + vn_gbm_graph_stage1.py + deps).
REPO_URL = 'https://github.com/ntquy9901/stock_vol_prediction01.git'
import os, shutil
if os.path.isdir('/content/repo'):
    shutil.rmtree('/content/repo')
get_ipython().system(f'git clone --depth 1 {REPO_URL} /content/repo')
for p in ('scripts/eda/gnnhar_sp500.py', 'scripts/eda/full_matrix.py', 'scripts/eda/vn_gbm_graph_stage1.py', 'results/gamma_gbm/sp500_sectors.json', 'results/gamma_gbm/sp500_earnings.parquet'):
    print(('HAS ' if os.path.exists('/content/repo/' + p) else 'MISSING ') + p)

In [ ]:
# 2. DATA from Google Drive (SP500 = Yahoo, redistribution-restricted, NOT on git). Tries known folder layouts.
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = ['/content/drive/MyDrive/luanvan_data/colab_bundle_sp500_clean.zip',
              '/content/drive/MyDrive/public_bk/luanvan_data/colab_bundle_sp500_clean.zip']
DRIVE_ZIP = next((c for c in CANDIDATES if os.path.exists(c)), None)
assert DRIVE_ZIP, 'bundle not found; set DRIVE_ZIP manually. Tried: ' + str(CANDIDATES)
print('using', DRIVE_ZIP)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    members = [m for m in z.namelist() if m.startswith('data/processed_enriched/')]
    z.extractall('/content/repo', members)
print('unpacked', len(members), 'data files')
get_ipython().system('ls /content/repo/data/processed_enriched')

In [ ]:
# 3. Deps. Colab ships torch+CUDA; the driver is pure torch (no torch_geometric).
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
get_ipython().system('pip -q install pandas numpy scikit-learn pyarrow')

In [ ]:
# 4. Run GNNHAR (GPU). SMOKE=True first to verify fast; then SMOKE=False for the full run.
import subprocess, time
SMOKE = True
cmd = ['python', 'scripts/eda/gnnhar_sp500.py'] + (['--smoke'] if SMOKE else [])
t0 = time.time()
p = subprocess.Popen(cmd, cwd='/content/repo', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
p.wait()
print('exit=' + str(p.returncode) + '  ' + format((time.time() - t0) / 60, '.1f') + ' min')

In [ ]:
# 5. Inspect + DOWNLOAD the result JSON (no git-push: keep the local overfit-evidence gate authoritative).
import glob, json
res = sorted(glob.glob('/content/repo/results/gnnhar/*.json'))
print('results:', res)
assert res, 'no result JSON -- did cell 4 finish?'
for f in res:
    print('==', f.split('/')[-1], '==')
    print(json.dumps(json.load(open(f)), indent=2)[:1500])
from google.colab import files
for f in res:
    files.download(f)

**Why A100 here and not for the GBM probes:** GNNHAR trains a graph neural net and is GPU-bound. The earnings-jitter / graph-additions probes are sklearn HistGradientBoostingRegressor (CPU-bound, already OpenMP across all cores); A100 does not help them.

**Results:** downloaded to your machine; drop into results/gnnhar/ locally, where the overfit-evidence pre-push gate applies before committing (do not push result JSONs straight from Colab).